# NeuroFinance AI — Phase 3 & 4: Data Preprocessing & Historical Features
This notebook inspects the preprocessed datasets generated by `preprocessing/preprocess.py` and verifies that no data leakage occurred during the scaling and imputation process.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
print("Libraries imported successfully!")

## 1. Load Processed Datasets

In [ ]:
train_path = "../data/processed/train.csv"
val_path = "../data/processed/validation.csv"
test_path = "../data/processed/test.csv"

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

print(f"Train Shape: {df_train.shape}")
print(f"Validation Shape: {df_val.shape}")
print(f"Test Shape: {df_test.shape}")

## 2. Verify Imputation and Formatting
Let's confirm there are no missing values remaining in the features and that all columns (except target and client ID) are numerical.

In [ ]:
# Check for null values
null_count = df_train.isnull().sum().sum()
print(f"Total Null Values in Processed Train Set: {null_count}")

# Check data types of features
feature_cols = df_train.drop(columns=["SK_ID_CURR", "TARGET"]).columns
non_numeric_cols = df_train[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric feature columns: {len(non_numeric_cols)}")

## 3. Verify Class Ratios Across Splits

In [ ]:
print(f"Train Default Rate: {df_train['TARGET'].mean():.4%}")
print(f"Validation Default Rate: {df_val['TARGET'].mean():.4%}")
print(f"Test Default Rate: {df_test['TARGET'].mean():.4%}")

## 4. Test Loading Preprocessor Pipeline
Let's make sure we can load the preprocessor from `models/preprocessor.pkl` and use it on a mock raw record. This verifies that our online inference pipeline will work correctly.

In [ ]:
with open("../models/preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

print("Preprocessor loaded successfully!")
print(f"Total processed numerical columns: {len(preprocessor.num_cols)}")
print(f"Total processed categorical columns: {len(preprocessor.cat_cols)}")
print(f"Total output features after encoding: {len(preprocessor.all_processed_cols)}")

# Create a mock raw customer record (from first training raw application)
raw_app = pd.read_csv("../data/raw/application_train.csv", nrows=1).drop(columns=["TARGET"])
print(f"Mock Raw Shape: {raw_app.shape}")

# Run base engineering on the single record
from preprocessing.preprocess import add_financial_features
mock_engineered = add_financial_features(raw_app)

# Aggregate historical features for this customer
# (Here we will just add mock placeholder columns for historical features to match output of preprocessing.preprocess)
hist_cols = ['previous_loan_count', 'active_loan_count', 'previous_application_count',
             'previous_approval_rate', 'previous_credit_amount', 'average_payment_delay',
             'late_payment_count', 'credit_utilization']
for c in hist_cols:
    mock_engineered[c] = 0.0

# Transform
mock_transformed = preprocessor.transform(mock_engineered)
print(f"Transformed Mock Record Shape: {mock_transformed.shape}")
print(mock_transformed.iloc[:, :10])